# VitalDB — Postoperative ICU Admission Prediction

Predicts postoperative ICU admission (`icu_days > 0`) using preoperative clinical features, intraoperative vital signs, and intraoperative waveform signals (ECG and PPG morphology features).

**Methodology:**
- 80/20 stratified holdout split performed before any cross-validation or model fitting
- 5-fold stratified cross-validation on the 80% training set for model evaluation and threshold selection
- SMOTE applied within training folds only (never before the train/val split, never on the holdout)
- Bootstrap confidence intervals (n=2000) computed on holdout predictions
- Six-model comparison structure: Signal only → Clinical only → Vitals only → Signal+Clinical → Clinical+Vitals → Full

In [ ]:
import pandas as pd
import numpy as np
import vitaldb
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score
)
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from scipy.stats import skew, kurtosis
import shap
import matplotlib as mpl
import matplotlib.pyplot as plt
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import json, warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load Clinical Data

In [ ]:
df = pd.read_csv('clinical_data.csv')
print(f"Full cohort: {df.shape}")
print(f"ICU admissions: {(df['icu_days'] > 0).sum()} ({(df['icu_days'] > 0).mean()*100:.1f}%)")

## 2. Feature Engineering

In [ ]:
data = df.copy()
data['icu_admit'] = (data['icu_days'] > 0).astype(int)
data['ecg_abnormal'] = (data['preop_ecg'] != 'Normal Sinus Rhythm').astype(int)

for col in ['sex', 'department', 'optype']:
    if col in data.columns:
        data[col] = LabelEncoder().fit_transform(data[col].astype(str))

FEATURES_CLINICAL = [
    'age', 'sex', 'bmi', 'asa', 'emop',
    'preop_htn', 'preop_dm', 'ecg_abnormal',
    'preop_hb', 'preop_plt', 'preop_pt', 'preop_aptt',
    'preop_na', 'preop_k', 'preop_gluc', 'preop_alb',
    'preop_cr', 'preop_bun', 'preop_ast', 'preop_alt'
]
FEATURES_VITALS = ['hr_mean', 'hr_min', 'hr_max', 'hr_std', 'spo2_mean', 'spo2_min']
FEATURES_PPG = [
    'ppg_mean', 'ppg_median', 'ppg_std', 'ppg_iqr', 'ppg_range',
    'ppg_skew', 'ppg_kurtosis', 'ppg_energy', 'ppg_mad', 'ppg_min', 'ppg_max'
]
FEATURES_ECG = [
    'ecg_mean', 'ecg_median', 'ecg_std', 'ecg_iqr', 'ecg_range',
    'ecg_skew', 'ecg_kurtosis', 'ecg_energy', 'ecg_mad', 'ecg_min', 'ecg_max'
]

print(f"Clinical features: {len(FEATURES_CLINICAL)}")
print(f"Vitals features: {len(FEATURES_VITALS)}")
print(f"PPG features: {len(FEATURES_PPG)}")
print(f"ECG features: {len(FEATURES_ECG)}")

## 3. Waveform Feature Extraction

Extracts statistical morphology features (mean, median, std, IQR, range, skew, kurtosis, energy, mean absolute difference, min, max) from raw PPG and ECG Lead II waveforms via the VitalDB API. Simple morphology statistics were used instead of clinical interval measurements (RR, PR, QRS, QT) because automated wave delineation on intraoperative ECG yielded usable intervals on only ~25% of cases — too low a yield to support modeling without selection bias. See Methods for full discussion.

In [ ]:
def extract_ppg_features(caseid):
    try:
        vals = vitaldb.load_case(caseid, ['SNUADC/PLETH'])
        ppg = vals[:, 0]
        ppg = ppg[~np.isnan(ppg)]
        if len(ppg) < 1000:
            return None
        p1, p99 = np.percentile(ppg, 1), np.percentile(ppg, 99)
        ppg = np.clip(ppg, p1, p99).astype(float)
        return {
            'caseid': caseid, 'ppg_mean': np.mean(ppg), 'ppg_median': np.median(ppg),
            'ppg_std': np.std(ppg), 'ppg_iqr': np.percentile(ppg, 75) - np.percentile(ppg, 25),
            'ppg_range': np.max(ppg) - np.min(ppg), 'ppg_skew': skew(ppg),
            'ppg_kurtosis': kurtosis(ppg), 'ppg_energy': np.mean(ppg**2),
            'ppg_mad': np.mean(np.abs(np.diff(ppg))), 'ppg_min': np.min(ppg), 'ppg_max': np.max(ppg),
        }
    except:
        return None


def extract_ecg_features(caseid):
    try:
        vals = vitaldb.load_case(caseid, ['SNUADC/ECG_II'])
        ecg = vals[:, 0]
        ecg = ecg[~np.isnan(ecg)]
        if len(ecg) < 1000:
            return None
        p1, p99 = np.percentile(ecg, 1), np.percentile(ecg, 99)
        ecg = np.clip(ecg, p1, p99).astype(float)
        return {
            'caseid': caseid, 'ecg_mean': np.mean(ecg), 'ecg_median': np.median(ecg),
            'ecg_std': np.std(ecg), 'ecg_iqr': np.percentile(ecg, 75) - np.percentile(ecg, 25),
            'ecg_range': np.max(ecg) - np.min(ecg), 'ecg_skew': skew(ecg),
            'ecg_kurtosis': kurtosis(ecg), 'ecg_energy': np.mean(ecg**2),
            'ecg_mad': np.mean(np.abs(np.diff(ecg))), 'ecg_min': np.min(ecg), 'ecg_max': np.max(ecg),
        }
    except:
        return None


def extract_vitals(caseid):
    try:
        vals = vitaldb.load_case(caseid, ['Solar8000/HR', 'Solar8000/PLETH_SPO2'])
        hr   = vals[:, 0]; hr   = hr[~np.isnan(hr)]
        spo2 = vals[:, 1]; spo2 = spo2[~np.isnan(spo2)]
        row = {'caseid': caseid}
        row.update({'hr_mean': np.mean(hr), 'hr_min': np.min(hr), 'hr_max': np.max(hr), 'hr_std': np.std(hr)}
                   if len(hr) > 10 else {'hr_mean': np.nan, 'hr_min': np.nan, 'hr_max': np.nan, 'hr_std': np.nan})
        row.update({'spo2_mean': np.mean(spo2), 'spo2_min': np.min(spo2)}
                   if len(spo2) > 10 else {'spo2_mean': np.nan, 'spo2_min': np.nan})
        return row
    except:
        return {'caseid': caseid, 'hr_mean': np.nan, 'hr_min': np.nan, 'hr_max': np.nan,
                'hr_std': np.nan, 'spo2_mean': np.nan, 'spo2_min': np.nan}


# Run extraction (uncomment to re-extract; otherwise load from saved CSVs in next cell)
# all_caseids = data['caseid'].tolist()
# ppg_results, ecg_results, vitals_results = [], [], []
# with ThreadPoolExecutor(max_workers=8) as executor:
#     futures_ppg    = {executor.submit(extract_ppg_features, cid): cid for cid in all_caseids}
#     futures_ecg    = {executor.submit(extract_ecg_features, cid): cid for cid in all_caseids}
#     futures_vitals = {executor.submit(extract_vitals, cid): cid for cid in all_caseids}
#     for f in tqdm(as_completed(futures_ppg), total=len(all_caseids), desc='PPG'):
#         r = f.result()
#         if r: ppg_results.append(r)
#     for f in tqdm(as_completed(futures_ecg), total=len(all_caseids), desc='ECG'):
#         r = f.result()
#         if r: ecg_results.append(r)
#     for f in tqdm(as_completed(futures_vitals), total=len(all_caseids), desc='Vitals'):
#         vitals_results.append(f.result())
# pd.DataFrame(ppg_results).to_csv('vitaldb_ppg_features.csv', index=False)
# pd.DataFrame(ecg_results).to_csv('vitaldb_ecg_features.csv', index=False)
# pd.DataFrame(vitals_results).to_csv('vitaldb_hr_spo2.csv', index=False)

ppg_df    = pd.read_csv('vitaldb_ppg_features.csv')
ecg_df    = pd.read_csv('vitaldb_ecg_features.csv')
vitals_df = pd.read_csv('vitaldb_hr_spo2.csv')
print(f"PPG: {len(ppg_df)}, ECG: {len(ecg_df)}, Vitals: {len(vitals_df)}")

## 4. Shared Pipeline Functions

In [ ]:
def prep_X(df, cols):
    X = df[cols].copy().apply(pd.to_numeric, errors='coerce')
    X = X.replace([np.inf, -np.inf], np.nan)
    for col in X.columns:
        X[col] = X[col].fillna(X[col].median())
    return X


def run_cv_smote(X, y, label):
    cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    smote = SMOTE(random_state=RANDOM_STATE)
    oof   = np.zeros(len(y))
    fold_rocs, fold_prs = [], []
    for tr_idx, val_idx in cv.split(X, y):
        X_tr, y_tr   = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)
        m = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03,
                          subsample=0.8, colsample_bytree=0.8,
                          random_state=RANDOM_STATE, verbosity=0)
        m.fit(X_tr_sm, y_tr_sm)
        preds = m.predict_proba(X_val)[:, 1]
        oof[val_idx] = preds
        fold_rocs.append(roc_auc_score(y_val, preds))
        fold_prs.append(average_precision_score(y_val, preds))
    cv_roc = roc_auc_score(y, oof)
    cv_pr  = average_precision_score(y, oof)
    thresholds = np.linspace(0.01, 0.99, 200)
    f1s = [f1_score(y, (oof >= t).astype(int), zero_division=0) for t in thresholds]
    best_thresh = thresholds[np.argmax(f1s)]
    print(f"{label}: CV AUC-ROC {cv_roc:.4f} \u00b1 {np.std(fold_rocs):.4f} | CV AUC-PR {cv_pr:.4f} \u00b1 {np.std(fold_prs):.4f}")
    X_sm, y_sm = smote.fit_resample(X, y)
    final = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03,
                          subsample=0.8, colsample_bytree=0.8,
                          random_state=RANDOM_STATE, verbosity=0)
    final.fit(X_sm, y_sm)
    return final, best_thresh, cv_roc, cv_pr


def bootstrap_ci(y_true, y_pred, metric_fn, n=2000):
    scores = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        scores.append(metric_fn(y_true[idx], y_pred[idx]))
    return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)


def bootstrap_delta(y_true, pred_a, pred_b, n=2000):
    deltas = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        deltas.append(roc_auc_score(y_true[idx], pred_b[idx]) - roc_auc_score(y_true[idx], pred_a[idx]))
    return np.mean(deltas), np.percentile(deltas, 2.5), np.percentile(deltas, 97.5)


def eval_holdout(model, X_test, y_test, best_thresh, label):
    preds = model.predict_proba(X_test)[:, 1]
    y_arr = np.array(y_test)
    roc_mean, roc_lo, roc_hi = bootstrap_ci(y_arr, preds, roc_auc_score)
    pr_mean,  pr_lo,  pr_hi  = bootstrap_ci(y_arr, preds, average_precision_score)
    y_bin = (preds >= best_thresh).astype(int)
    f1    = f1_score(y_arr, y_bin, zero_division=0)
    prec  = precision_score(y_arr, y_bin, zero_division=0)
    rec   = recall_score(y_arr, y_bin, zero_division=0)
    print(f"{label} \u2014 HOLDOUT: AUC-ROC {roc_mean:.4f} ({roc_lo:.4f}-{roc_hi:.4f}) | AUC-PR {pr_mean:.4f} ({pr_lo:.4f}-{pr_hi:.4f}) | F1 {f1:.4f} | Prec {prec:.4f} | Rec {rec:.4f}")
    return preds, {'roc': roc_mean, 'roc_ci': (roc_lo, roc_hi), 'pr': pr_mean, 'pr_ci': (pr_lo, pr_hi),
                   'f1': f1, 'precision': prec, 'recall': rec, 'threshold': best_thresh}

print("Pipeline functions defined.")

## 5. ECG Analysis — Six-Model Comparison

Order: Signal only → Clinical only → Vitals only → Signal+Clinical → Clinical+Vitals → Full

In [ ]:
merged_ecg = data.merge(vitals_df, on='caseid', how='inner').merge(ecg_df, on='caseid', how='inner')
print(f"ECG cohort: {len(merged_ecg)} | ICU+: {merged_ecg['icu_admit'].sum()} ({merged_ecg['icu_admit'].mean()*100:.1f}%)")

train_ecg, test_ecg = train_test_split(merged_ecg, test_size=0.2, stratify=merged_ecg['icu_admit'], random_state=RANDOM_STATE)
train_ecg = train_ecg.reset_index(drop=True)
test_ecg  = test_ecg.reset_index(drop=True)

y_train_ecg = train_ecg['icu_admit'].astype(int)
y_test_ecg  = test_ecg['icu_admit'].astype(int)

ecg_models = {}
ecg_results = {}

for name, feats in [
    ('ecg_only', FEATURES_ECG),
    ('clinical_only', FEATURES_CLINICAL),
    ('vitals_only', FEATURES_VITALS),
    ('ecg_clinical', FEATURES_ECG + FEATURES_CLINICAL),
    ('clinical_vitals', FEATURES_CLINICAL + FEATURES_VITALS),
    ('full', FEATURES_ECG + FEATURES_CLINICAL + FEATURES_VITALS),
]:
    X_tr = prep_X(train_ecg, feats)
    X_te = prep_X(test_ecg, feats)
    model, thresh, cv_roc, cv_pr = run_cv_smote(X_tr, y_train_ecg, name)
    preds, res = eval_holdout(model, X_te, y_test_ecg, thresh, name)
    res['cv_roc'] = cv_roc
    res['cv_pr']  = cv_pr
    ecg_models[name]  = model
    ecg_results[name] = {'preds': preds, 'metrics': res}

## 6. PPG Analysis — Six-Model Comparison

In [ ]:
merged_ppg = data.merge(vitals_df, on='caseid', how='inner').merge(ppg_df, on='caseid', how='inner')
print(f"PPG cohort: {len(merged_ppg)} | ICU+: {merged_ppg['icu_admit'].sum()} ({merged_ppg['icu_admit'].mean()*100:.1f}%)")

train_ppg, test_ppg = train_test_split(merged_ppg, test_size=0.2, stratify=merged_ppg['icu_admit'], random_state=RANDOM_STATE)
train_ppg = train_ppg.reset_index(drop=True)
test_ppg  = test_ppg.reset_index(drop=True)

y_train_ppg = train_ppg['icu_admit'].astype(int)
y_test_ppg  = test_ppg['icu_admit'].astype(int)

ppg_models = {}
ppg_results = {}

for name, feats in [
    ('ppg_only', FEATURES_PPG),
    ('clinical_only', FEATURES_CLINICAL),
    ('vitals_only', FEATURES_VITALS),
    ('ppg_clinical', FEATURES_PPG + FEATURES_CLINICAL),
    ('clinical_vitals', FEATURES_CLINICAL + FEATURES_VITALS),
    ('full', FEATURES_PPG + FEATURES_CLINICAL + FEATURES_VITALS),
]:
    X_tr = prep_X(train_ppg, feats)
    X_te = prep_X(test_ppg, feats)
    model, thresh, cv_roc, cv_pr = run_cv_smote(X_tr, y_train_ppg, name)
    preds, res = eval_holdout(model, X_te, y_test_ppg, thresh, name)
    res['cv_roc'] = cv_roc
    res['cv_pr']  = cv_pr
    ppg_models[name]  = model
    ppg_results[name] = {'preds': preds, 'metrics': res}

## 7. Incremental Comparisons (Bootstrap Delta AUC)

In [ ]:
def print_deltas(results, y_test, signal_name):
    y_arr = np.array(y_test)
    d1 = bootstrap_delta(y_arr, results['clinical_only']['preds'], results[f'{signal_name}_clinical']['preds'])
    d2 = bootstrap_delta(y_arr, results[f'{signal_name}_only']['preds'], results[f'{signal_name}_clinical']['preds'])
    d3 = bootstrap_delta(y_arr, results['clinical_vitals']['preds'], results[f'{signal_name}_clinical']['preds'])
    d4 = bootstrap_delta(y_arr, results[f'{signal_name}_clinical']['preds'], results['full']['preds'])
    d5 = bootstrap_delta(y_arr, results['clinical_vitals']['preds'], results['full']['preds'])
    print(f"\n{signal_name.upper()} deltas:")
    print(f"  Signal+Clin over Clin alone:        {d1[0]:+.4f} ({d1[1]:+.4f} to {d1[2]:+.4f})")
    print(f"  Signal+Clin over Signal alone:       {d2[0]:+.4f} ({d2[1]:+.4f} to {d2[2]:+.4f})")
    print(f"  Signal+Clin vs Vitals+Clin:          {d3[0]:+.4f} ({d3[1]:+.4f} to {d3[2]:+.4f})")
    print(f"  Full over Signal+Clin (vitals' val): {d4[0]:+.4f} ({d4[1]:+.4f} to {d4[2]:+.4f})")
    print(f"  Full over Clin+Vitals (signal's val): {d5[0]:+.4f} ({d5[1]:+.4f} to {d5[2]:+.4f})")

print_deltas(ecg_results, y_test_ecg, 'ecg')
print_deltas(ppg_results, y_test_ppg, 'ppg')

## 8. Save Results

In [ ]:
def serialize(results):
    out = {}
    for name, r in results.items():
        m = r['metrics'].copy()
        out[name] = m
    return out

all_results = {
    'ecg': serialize(ecg_results),
    'ppg': serialize(ppg_results),
}
with open('vitaldb_icu_admission_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print("Saved to vitaldb_icu_admission_results.json")

## 9. Figures: ROC/PR Curves

In [ ]:
mpl.rcParams.update({
    'font.family': 'Arial', 'font.size': 11, 'axes.linewidth': 1.2,
    'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'figure.dpi': 300,
})

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

for col, (results, y_test, label) in enumerate([(ecg_results, y_test_ecg, 'ECG'), (ppg_results, y_test_ppg, 'PPG')]):
    ax = axes[0, col]
    for name, color in [('full', '#2ECC71'), ('clinical_vitals', '#D65F5F'), (f'{label.lower()}_clinical', '#4878CF')]:
        preds = results[name]['preds']
        auc = results[name]['metrics']['roc']
        fpr, tpr, _ = roc_curve(y_test, preds)
        ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc:.3f})')
    ax.plot([0,1],[0,1],'--',lw=1.2,color='#cccccc')
    ax.set_xlabel('1 - Specificity'); ax.set_ylabel('Sensitivity')
    ax.set_title(f'ROC \u2014 {label} Models', fontweight='bold')
    ax.legend(loc='lower right', fontsize=8)

    ax2 = axes[1, col]
    for name, color in [('full', '#2ECC71'), ('clinical_vitals', '#D65F5F'), (f'{label.lower()}_clinical', '#4878CF')]:
        preds = results[name]['preds']
        ap = results[name]['metrics']['pr']
        prec, rec, _ = precision_recall_curve(y_test, preds)
        ax2.plot(rec, prec, lw=2, color=color, label=f'{name} (AP={ap:.3f})')
    ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
    ax2.set_title(f'PR \u2014 {label} Models', fontweight='bold')
    ax2.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('vitaldb_icu_admission_roc_pr.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()